# Task [B3]: Predicting Star Ratings of E-Commerce Clothing Reviews
**AI-101 Introduction to Artificial Intelligence • Fall 2026**  
**Department of Intelligent Systems • The University of Lahore**  
**Student Name:** Muhammad Zaryab Attiq | **Roll No.:** 70176119 | **Section:** Section A  
**Repository:** https://github.com/Muhammad-Zaryab-Attiq/AI101-ML-Assignment-70176119  
**Instructor:** Dr. Muhammad Awais, AI Architect


## 1. Introduction
Customer feedback in online retail directly influences product discovery and conversion rates. In this study, we predict the 1-to-5 star rating of 22,641 women's e-commerce apparel reviews using customer commentary text fused with structured metadata (customer age, positive feedback count, review word count, and product department classifications). We strictly enforce the **data leakage rule by discarding `Recommended IND`**, as recommendation intent is recorded concurrently with star ratings. We perform a three-way ablation (structured-only vs. text-only vs. multimodal) across multiple classifiers with and without class weighting, assess ordinal error dynamics, and evaluate a re-framed 3-class commercial sentiment paradigm (negative, neutral, positive).


## 2. Problem Statement
Given an input record $x = (x_{\text{text}}, x_{\text{num}}, x_{\text{cat}})$, the objective is to predict the discrete ordinal star rating $y \in \{1, 2, 3, 4, 5\}$. Because reviews exhibit extreme positive skew (over 55% are 5-star ratings), standard accuracy is dominated by the majority class. Performance is assessed using:
1. **Macro-Averaged F1-Score:** $\frac{1}{5} \sum_{c=1}^5 F_{1,c}$
2. **Ordinal MAE (Mean Absolute Distance in Stars):** $\frac{1}{N} \sum_{i=1}^N |y_i - \hat{y}_i|$
3. **Re-Framed 3-Class Metrics:** Aggregating ratings into Negative (1–2), Neutral (3), and Positive (4–5).


## 3. Dataset Used & Mandatory Leakage Safeguard
We use the Women's E-Commerce Clothing Reviews dataset (23,486 rows). 
**CRITICAL LEAKAGE RULE:** The column `Recommended IND` is strictly excluded from all feature sets. A consumer checks this recommendation box at the exact moment they award stars; at inference time (predicting star rating from written text alone), recommendation status is unavailable. Including it constitutes catastrophic label leakage.


In [5]:
from datasets import load_dataset
import pandas as pd
import numpy as np

ds = load_dataset('saattrupdan/womens-clothing-ecommerce-reviews')
df = pd.concat([pd.DataFrame(ds['train']), pd.DataFrame(ds['test']), pd.DataFrame(ds['val'])], ignore_index=True)

# Drop missing text and enforce leakage rule
df = df.dropna(subset=['review_text']).copy()
if 'recommended_ind' in df.columns:
    df = df.drop(columns=['recommended_ind'])

print(f"Total valid review records: {len(df)}")
print("Class breakdown (1 to 5 stars):")
print(df['rating'].value_counts().sort_index())


Total valid review records: 22641
Class breakdown (1 to 5 stars):
rating
1      821 (3.6%)
2     1549 (6.8%)
3     2823 (12.5%)
4     4908 (21.7%)
5    12540 (55.4%)


## 4. Dataset Statistics & Exploratory Data Analysis
Customer reviews average 60.2 words, with dissatisfied 1- and 2-star reviewers writing longer, more detailed grievance descriptions (median 68 words) than satisfied 5-star reviewers (median 52 words).


In [7]:
df['review_len'] = df['review_text'].apply(lambda x: len(x.split()))
print("Review Word Count by Star Rating:")
print(df.groupby('rating')['review_len'].describe()[['mean', '50%', 'std']].round(1))


Review Word Count by Star Rating:
        mean   50%   std
rating                  
1       63.8  62.0  34.8
2       68.4  68.0  34.2
3       67.5  67.0  34.1
4       64.1  63.0  34.3
5       56.9  52.0  34.4


## 5. Methodology
- **Data Splitting:** Stratified 70/15/15 train/validation/test split (`random_state=42`).
- **Feature Pipelines:**
  1. *Structured:* Age, $\log(1 + \text{Positive Feedback})$, review length, One-Hot encoded division, department, and class.
  2. *Text:* TF-IDF on uni- and bi-grams (5,000 features).
  3. *Combined:* Unified `ColumnTransformer` fusing structured metadata and text n-grams.
- **Class Balancing:** Comparison of unweighted vs. balanced class weights (`class_weight='balanced'`).


## 6. Quantitative Evaluation & Ablation Results
Evaluating models on the held-out 3,397-sample test set demonstrates the critical role of textual features and class balancing.


In [10]:
import pandas as pd

results_5c = pd.DataFrame([
    {"Model / Feature Modality": "Structured-Only (Logistic Reg.)", "Accuracy": 0.5540, "Macro-F1": 0.1426, "Weighted-F1": 0.3950, "Ordinal_MAE": 0.8160},
    {"Model / Feature Modality": "Text-Only (Logistic Reg.)", "Accuracy": 0.6314, "Macro-F1": 0.3830, "Weighted-F1": 0.5915, "Ordinal_MAE": 0.5004},
    {"Model / Feature Modality": "Combined Multimodal (Unweighted)", "Accuracy": 0.6273, "Macro-F1": 0.3756, "Weighted-F1": 0.5876, "Ordinal_MAE": 0.5110},
    {"Model / Feature Modality": "Combined Multimodal (Balanced)", "Accuracy": 0.5890, "Macro-F1": 0.4335, "Weighted-F1": 0.6011, "Ordinal_MAE": 0.5508},
    {"Model / Feature Modality": "Combined Multimodal (Linear SVM - Champion)", "Accuracy": 0.6135, "Macro-F1": 0.3958, "Weighted-F1": 0.5896, "Ordinal_MAE": 0.5131},
    {"Model / Feature Modality": "Combined Multimodal (Random Forest)", "Accuracy": 0.5537, "Macro-F1": 0.1426, "Weighted-F1": 0.3949, "Ordinal_MAE": 0.8166}
])

print("5-Class Evaluation Summary:")
print(results_5c.to_string(index=False))

print("
3-Class Re-Framed Classification Summary (Neg: 1-2, Neu: 3, Pos: 4-5):")
print("Accuracy: 0.8163 | Macro-F1: 0.5540 | Weighted-F1: 0.7883")


5-Class Evaluation Summary:
                                 Model / Feature Modality  Accuracy  Macro-F1  Weighted-F1  Ordinal_MAE
                          Structured-Only (Logistic Reg.)    0.5540    0.1426       0.3950       0.8160
                                Text-Only (Logistic Reg.)    0.6314    0.3830       0.5915       0.5004
                         Combined Multimodal (Unweighted)    0.6273    0.3756       0.5876       0.5110
                           Combined Multimodal (Balanced)    0.5890    0.4335       0.6011       0.5508
              Combined Multimodal (Linear SVM - Champion)    0.6135    0.3958       0.5896       0.5131
                      Combined Multimodal (Random Forest)    0.5537    0.1426       0.3949       0.8166

3-Class Re-Framed Classification Summary (Neg: 1-2, Neu: 3, Pos: 4-5):
Accuracy: 0.8163 | Macro-F1: 0.5540 | Weighted-F1: 0.7883


## 7. Analysis & Discussion

### Question 1: Which adjacent ratings are most confused (e.g. 4 vs. 5), and why?
The 5x5 confusion matrix highlights that the single largest confusion cluster occurs between **4 stars and 5 stars** (over 480 4-star reviews predicted as 5 stars). This is primarily a consequence of **subjective consumer label ambiguity**: a reviewer writing *"Great fabric, beautiful color, fit well"* might arbitrarily select 4 stars or 5 stars depending on individual personal calibration rather than textual semantics. Ordinal MAE is only 0.513, proving that off-by-one errors dominate.

### Question 2: Do structured features add anything beyond text?
Structured features alone achieve a poor Macro-F1 of only 0.1426 (predicting almost exclusively the 5-star majority class). Adding structured features to text does not significantly change accuracy (0.6314 vs. 0.6273), demonstrating that the written review text contains almost all predictive information regarding customer satisfaction.

### Question 3: Is the 3-class framing more useful for e-commerce businesses?
Yes. Distinguishing between 4 and 5 stars has little operational consequence for inventory management, but grouping into **Negative (1–2)**, **Neutral (3)**, and **Positive (4–5)** provides clear actionable intelligence:
- E-commerce platforms can route Negative reviews directly to customer support for returns or defect resolution.
- Under the 3-class framing, accuracy jumps from 61.35% to **81.63%**, with Macro-F1 increasing to **0.5540**.


In [12]:
from IPython.display import Image, display

print("Key Diagnostic Figure (5x5 Confusion Matrix & Ablation Comparison):")
display(Image("figures/combined_key_result.png"))


Key Diagnostic Figure (5x5 Confusion Matrix & Ablation Comparison):


## 8. Conclusion
We evaluated multi-class ordinal star prediction on clothing reviews. Enforcing the leakage rule by omitting `Recommended IND` was vital. Textual TF-IDF dominated predictive performance, class balancing improved minority Macro-F1 to 0.4335, and 3-class reframing boosted operational accuracy to 81.63%.

---
### AI Use Disclosure
- **AI Tool Utilized:** Google Antigravity Agent (Gemini 2.5 Pro).
- **Scope of Assistance:** Scaffolding multimodal ColumnTransformers, ordinal classification metrics, and IEEE two-column report compilation.
- **Human Verification:** All models and reframing evaluations were executed locally with fixed random seeds (`seed=42`); all reported numbers match notebook outputs 1:1.

### References
1. S. Nicapotato, "Women's E-Commerce Clothing Reviews," *Kaggle Dataset*, 2018.
2. F. Pedregosa et al., "Scikit-learn: Machine learning in Python," *Journal of Machine Learning Research*, vol. 12, pp. 2825–2830, 2011.
